In [3]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

STRUCTURE_SENSITIVITY = 3 
INITIAL_CAPITAL = 1000
LEVERAGE = 2

def fetch_and_clean_data(ticker, start, end, interval='1d'):
    df = yf.download(ticker, start=start, end=end, interval=interval, progress=False)
    if isinstance(df.columns, pd.MultiIndex): 
        df.columns = df.columns.droplevel(1)
    df = df.round(2).reset_index()
    df.columns.name = None
    df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None) 
    return df

print("Fetching market data...")
bull_market_1_daily = fetch_and_clean_data('BTC-USD', '2018-12-15', '2021-11-10', '1d')
bull_market_2_daily = fetch_and_clean_data('BTC-USD', '2022-11-21', '2025-10-06', '1d')
bull_market_1_monthly = fetch_and_clean_data('BTC-USD', '2018-12-15', '2021-11-10', '1mo')
bull_market_2_monthly = fetch_and_clean_data('BTC-USD', '2022-11-21', '2025-10-06', '1mo')

def get_structure_swings(daily_price, window):
    swings = []
    for i in range(window, len(daily_price) - window):
        high_slice = daily_price['High'].iloc[i-window : i+window+1]
        low_slice = daily_price['Low'].iloc[i-window : i+window+1]
        
        if daily_price['High'].iloc[i] == high_slice.max():
            swings.append({'Date': daily_price['Date'].iloc[i], 'Price': daily_price['High'].iloc[i], 'Type': 'High'})
        elif daily_price['Low'].iloc[i] == low_slice.min():
            swings.append({'Date': daily_price['Date'].iloc[i], 'Price': daily_price['Low'].iloc[i], 'Type': 'Low'})
            
    if not swings: return pd.DataFrame()
    
    swings_df = pd.DataFrame(swings).sort_values('Date')
    clean_swings = []
    last_type = None
    
    for _, swing in swings_df.iterrows():
        if swing['Type'] != last_type:
            clean_swings.append(swing)
            last_type = swing['Type']
        else:
            if swing['Type'] == 'High' and swing['Price'] > clean_swings[-1]['Price']:
                clean_swings[-1] = swing
            elif swing['Type'] == 'Low' and swing['Price'] < clean_swings[-1]['Price']:
                clean_swings[-1] = swing
                
    return pd.DataFrame(clean_swings)

swings_1 = get_structure_swings(bull_market_1_daily, STRUCTURE_SENSITIVITY)
swings_2 = get_structure_swings(bull_market_2_daily, STRUCTURE_SENSITIVITY)

def find_fvg(df):
    fvgs = []
    for i in range(2, len(df)):
        c1, c2, c3 = df.iloc[i-2], df.iloc[i-1], df.iloc[i]
        activation_date = c3['Date'] + pd.DateOffset(months=1)
        if c3['Low'] > c1['High']:
            fvgs.append({'DrawDate': c2['Date'], 'Start': activation_date, 'Type': 'Bullish FVG', 'Top': c3['Low'], 'Bottom': c1['High']})
    return pd.DataFrame(fvgs)

base_fvg_1 = find_fvg(bull_market_1_monthly)
base_fvg_2 = find_fvg(bull_market_2_monthly)

def get_fvg_draw_segments(daily_price, fvg_base_df):
    segments = []
    if fvg_base_df.empty: return pd.DataFrame()
    
    for _, fvg in fvg_base_df.iterrows():
        curr_top = fvg['Top']
        curr_bottom = fvg['Bottom']
        draw_date = fvg['DrawDate'] 
        activation_date = fvg['Start']
        
        segments.append({
            'Start': draw_date, 'End': activation_date, 'Top': curr_top, 'Bottom': curr_bottom, 
            'Activation': activation_date, 'Type': fvg['Type']
        })
        
        valid_daily = daily_price[daily_price['Date'] >= activation_date]
        fully_filled = False
        start_draw = activation_date
        
        for i in range(len(valid_daily)):
            day = valid_daily.iloc[i]
            if day['Low'] < curr_top:
                if day['Date'] > start_draw:
                    segments.append({'Start': start_draw, 'End': day['Date'], 'Top': curr_top, 'Bottom': curr_bottom, 'Activation': activation_date, 'Type': fvg['Type']})
                
                if day['Low'] <= curr_bottom:
                    fully_filled = True
                    break 
                else:
                    curr_top = day['Low'] 
                    start_draw = day['Date']
                    
        if not fully_filled and not valid_daily.empty:
            segments.append({'Start': start_draw, 'End': daily_price['Date'].iloc[-1], 'Top': curr_top, 'Bottom': curr_bottom, 'Activation': activation_date, 'Type': fvg['Type']})
            
    return pd.DataFrame(segments)

fvg_segments_1 = get_fvg_draw_segments(bull_market_1_daily, base_fvg_1)
fvg_segments_2 = get_fvg_draw_segments(bull_market_2_daily, base_fvg_2)

def detect_mtfa_signals(daily_price, monthly_fvg, swing_window):
    signals = []
    if monthly_fvg.empty: return pd.DataFrame()
    
    true_highs = []
    for i in range(swing_window, len(daily_price) - swing_window):
        window_slice = daily_price['High'].iloc[i-swing_window : i+swing_window+1]
        if daily_price['High'].iloc[i] == window_slice.max():
            true_highs.append({'Date': daily_price['Date'].iloc[i], 'Price': daily_price['High'].iloc[i]})
    true_highs_df = pd.DataFrame(true_highs)
    
    for _, fvg in monthly_fvg.iterrows():
        valid_daily = daily_price[daily_price['Date'] >= fvg['Start']]
        if valid_daily.empty: continue
            
        in_zone = False
        for i in range(len(valid_daily)):
            curr_day = valid_daily.iloc[i]
            
            if not in_zone and curr_day['Low'] <= fvg['Top']:
                in_zone = True
                
            if in_zone:
                past_highs = true_highs_df[true_highs_df['Date'] < curr_day['Date']]
                if not past_highs.empty:
                    last_true_high = past_highs.iloc[-1]['Price']
                    if curr_day['Close'] > last_true_high:
                        signals.append({'Date': curr_day['Date'], 'Price': curr_day['Close'], 'FVG_Top': fvg['Top'], 'Type': 'LONG'})
                        break 
                        
    signals_df = pd.DataFrame(signals)
    
    if not signals_df.empty:
        signals_df = signals_df.drop_duplicates(subset=['Date']).reset_index(drop=True)
        
    return signals_df

signals_1 = detect_mtfa_signals(bull_market_1_daily, base_fvg_1, STRUCTURE_SENSITIVITY)
signals_2 = detect_mtfa_signals(bull_market_2_daily, base_fvg_2, STRUCTURE_SENSITIVITY)

def backtest_strategy(signals_df, daily_price, initial_capital, leverage):
    results = []
    equity_timeline = []
    if signals_df.empty: return pd.DataFrame(), pd.DataFrame(), 0, 0
        
    end_date = daily_price['Date'].iloc[-1]
    final_close_price = daily_price['Close'].iloc[-1]
    total_invested = 0
    total_pnl = 0
    
    trades = []

    for _, signal in signals_df.iterrows():
        entry_date = signal['Date']
        entry_price = signal['Price']
        total_invested += initial_capital 
        
        future_data = daily_price[daily_price['Date'] >= entry_date]
        if future_data.empty: continue
            
        liquidation_price = entry_price * (1 - (1 / leverage))
        lowest_point = future_data['Low'].min()
        
        if lowest_point <= liquidation_price:
            liq_day = future_data[future_data['Low'] <= liquidation_price].iloc[0]
            liq_date = liq_day['Date']
            pnl = -initial_capital
            days_held = (liq_date - entry_date).days
            
            trades.append({'entry_date': entry_date, 'entry_price': entry_price, 'exit_date': liq_date, 'pnl': pnl})
            
            results.append({
                'Entry Date': entry_date.strftime('%Y-%m-%d'),
                'Entry Price ($)': round(entry_price, 2),
                'Liquidation Price ($)': round(liquidation_price, 2),
                'Status': 'Liquidated',
                'Exit Date': liq_date.strftime('%Y-%m-%d'),
                'Investment Period (Days)': days_held,
                'Exit Price ($)': round(liquidation_price, 2),
                'ROI (%)': -100.0,
                'Profit/Loss ($)': pnl
            })
        else:
            price_change_pct = (final_close_price - entry_price) / entry_price
            leveraged_roi = price_change_pct * leverage * 100
            pnl = initial_capital * (leveraged_roi / 100)
            days_held = (end_date - entry_date).days
            
            trades.append({'entry_date': entry_date, 'entry_price': entry_price, 'exit_date': end_date, 'pnl': pnl})
            
            results.append({
                'Entry Date': entry_date.strftime('%Y-%m-%d'),
                'Entry Price ($)': round(entry_price, 2),
                'Liquidation Price ($)': round(liquidation_price, 2),
                'Status': 'Success (Hold)',
                'Exit Date': end_date.strftime('%Y-%m-%d'),
                'Investment Period (Days)': days_held,
                'Exit Price ($)': round(final_close_price, 2),
                'ROI (%)': round(leveraged_roi, 2),
                'Profit/Loss ($)': round(pnl, 2)
            })
            
    for _, day in daily_price.iterrows():
        current_date = day['Date']
        current_price = day['Close']
        daily_pnl = 0
        
        for trade in trades:
            if current_date < trade['entry_date']:
                continue
            elif current_date >= trade['exit_date']:
                daily_pnl += trade['pnl']
            else:
                unrealized_change = (current_price - trade['entry_price']) / trade['entry_price']
                daily_pnl += initial_capital * (unrealized_change * leverage)
                
        equity_timeline.append({'Date': current_date, 'PnL': daily_pnl})

    results_df = pd.DataFrame(results)
    equity_df = pd.DataFrame(equity_timeline)
    
    if not results_df.empty:
        results_df.index = range(1, len(results_df) + 1)
        results_df.index.name = 'No.'
        total_pnl = results_df['Profit/Loss ($)'].sum()
        
    return results_df, equity_df, total_invested, total_pnl

def plot_structure_chart(daily_price, swings_df, title):
    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=daily_price['Date'], open=daily_price['Open'], high=daily_price['High'],
        low=daily_price['Low'], close=daily_price['Close'], name='Daily Price',
        increasing_line_color='#26A69A', decreasing_line_color='#EF5350'
    ))
    
    if not swings_df.empty:
        fig.add_trace(go.Scatter(
            x=swings_df['Date'], y=swings_df['Price'], mode='lines+markers',
            line=dict(color='#2196F3', width=2), marker=dict(size=8, color='#FFC107'),
            name='Structure'
        ))
    fig.update_layout(title=title, template='plotly_dark', xaxis_rangeslider_visible=False, height=600)
    fig.show()

def plot_final_chart(daily_price, fvg_segments, signals_df, title):
    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=daily_price['Date'], open=daily_price['Open'], high=daily_price['High'],
        low=daily_price['Low'], close=daily_price['Close'], name='Daily Price',
        increasing_line_color='#26A69A', decreasing_line_color='#EF5350'
    ))
    
    if not fvg_segments.empty:
        for _, seg in fvg_segments.iterrows():
            fig.add_shape(
                type="rect", x0=seg['Start'], y0=seg['Bottom'], x1=seg['End'], y1=seg['Top'],
                fillcolor='rgba(0, 200, 83, 0.15)', line=dict(color='rgba(0, 200, 83, 0.4)', width=1), layer="below"
            )
        unique_activations = fvg_segments['Activation'].unique()
        for act_date in unique_activations:
            fvg_data = fvg_segments[fvg_segments['Activation'] == act_date].iloc[0]
            fig.add_shape(
                type="line", x0=act_date, y0=fvg_data['Bottom'], x1=act_date, y1=fvg_data['Top'],
                line=dict(color='rgba(255, 255, 255, 0.5)', width=2, dash='dot'), layer="below"
            )
        
    if not signals_df.empty:
        fig.add_trace(go.Scatter(
            x=signals_df['Date'], y=signals_df['Price'], mode='markers',
            marker=dict(symbol='triangle-up', size=16, color='#FFD700', line=dict(width=2, color='white')),
            name='Long Entry', hovertemplate='LONG<br>Date: %{x}<br>Price: %{y}<extra></extra>'
        ))
    fig.update_layout(title=title, template='plotly_dark', xaxis_rangeslider_visible=False, height=800)
    fig.show()

def plot_equity_curve(equity_df, title):
    if equity_df.empty: return
    fig = go.Figure()
    colors = ['#00E676' if val >= 0 else '#FF1744' for val in equity_df['PnL']]
    fig.add_trace(go.Bar(
        x=equity_df['Date'], y=equity_df['PnL'], marker_color=colors, name='Net Profit/Loss ($)'
    ))
    fig.update_layout(title=title, template='plotly_dark', height=400, yaxis_title="Net Profit / Loss ($)", xaxis_title="Date")
    fig.show()

print("\nCalculations complete. Generating reports for Bull Market 1 (2018-2021)...")
plot_structure_chart(bull_market_1_daily, swings_1, "Bull Market 1 - Structure")
plot_final_chart(bull_market_1_daily, fvg_segments_1, signals_1, "Bull Market 1 - Long Entry")

results_1, equity_1, invested_1, pnl_1 = backtest_strategy(signals_1, bull_market_1_daily, INITIAL_CAPITAL, LEVERAGE)
if not results_1.empty:
    plot_equity_curve(equity_1, "Bull Market 1 - Account Balance (Profit/Loss over time)")
    print("\nPROFIT REPORT: BULL MARKET 1")
    display(results_1)
    total_roi_1 = (pnl_1 / invested_1) * 100
    print(f"Total invested: ${invested_1}")
    print(f"Net Profit/Loss (PnL): ${round(pnl_1, 2)}")
    print(f"TOTAL STRATEGY ROI: {round(total_roi_1, 2)}%")
    print(f"Total withdrawn from exchange: ${round(invested_1 + pnl_1, 2)}")
else:
    print("No signals to test for Bull Market 1.")

print("\nGenerating reports for Bull Market 2 (2022-2025)...")
plot_structure_chart(bull_market_2_daily, swings_2, "Bull Market 2 - Structure")
plot_final_chart(bull_market_2_daily, fvg_segments_2, signals_2, "Bull Market 2 - Long Entry")

results_2, equity_2, invested_2, pnl_2 = backtest_strategy(signals_2, bull_market_2_daily, INITIAL_CAPITAL, LEVERAGE)
if not results_2.empty:
    plot_equity_curve(equity_2, "Bull Market 2 - Account Balance (Profit/Loss over time)")
    print("\nPROFIT REPORT: BULL MARKET 2")
    display(results_2)
    total_roi_2 = (pnl_2 / invested_2) * 100
    print(f"Total invested: ${invested_2}")
    print(f"Net Profit/Loss (PnL): ${round(pnl_2, 2)}")
    print(f"TOTAL STRATEGY ROI: {round(total_roi_2, 2)}%")
    print(f"Total withdrawn from exchange: ${round(invested_2 + pnl_2, 2)}")
else:
    print("No signals to test for Bull Market 2.")

Fetching market data...

Calculations complete. Generating reports for Bull Market 1 (2018-2021)...



PROFIT REPORT: BULL MARKET 1


,Entry Date,Entry Price ($),Liquidation Price ($),Status,Exit Date,Investment Period (Days),Exit Price ($),ROI (%),Profit/Loss ($)
No.,,,,,,,,,
1,2020-04-06,7271.78,3635.89,Success (Hold),2021-11-09,582,66971.83,1641.97,16419.65
2,2019-10-25,8660.70,4330.35,Liquidated,2020-03-13,140,4330.35,-100.00,-1000.00
3,2019-10-09,8595.74,4297.87,Liquidated,2020-03-13,156,4297.87,-100.00,-1000.00
4,2020-10-09,11064.46,5532.23,Success (Hold),2021-11-09,396,66971.83,1010.58,10105.76
5,2021-06-14,40218.48,20109.24,Success (Hold),2021-11-09,148,66971.83,133.04,1330.40


Total invested: $5000
Net Profit/Loss (PnL): $25855.81
TOTAL STRATEGY ROI: 517.12%
Total withdrawn from exchange: $30855.81

Generating reports for Bull Market 2 (2022-2025)...



PROFIT REPORT: BULL MARKET 2


,Entry Date,Entry Price ($),Liquidation Price ($),Status,Exit Date,Investment Period (Days),Exit Price ($),ROI (%),Profit/Loss ($)
No.,,,,,,,,,
1,2023-03-13,24197.53,12098.76,Success (Hold),2025-10-05,937,123513.48,820.88,8208.77
2,2023-05-28,28085.65,14042.83,Success (Hold),2025-10-05,861,123513.48,679.55,6795.49
3,2024-05-15,66267.49,33133.75,Success (Hold),2025-10-05,508,123513.48,172.77,1727.72
4,2025-01-17,104462.04,52231.02,Success (Hold),2025-10-05,261,123513.48,36.48,364.75


Total invested: $4000
Net Profit/Loss (PnL): $17096.73
TOTAL STRATEGY ROI: 427.42%
Total withdrawn from exchange: $21096.73
